# The most widespread individual dark ORF

Every previous notebook grouped the dark proteome into **families** at 30% identity / 80%
coverage. This one drops to the literal protein: two dark ORFs are "the same" only if their
translations are **character-for-character identical**.

The question is simple — *which single unannotated protein sits on the most plasmids?* — and
the answer turns out to depend entirely on what "most" means. Both answers are computed here,
along with the sequences themselves.

Source data: `data/dark_orf_run/dark_orfs.faa` (378,552 dark ORFs from the 71,414 payload-free
small plasmids), produced by `scripts/extract_plasann_proteins.py`; families from
`scripts/cluster_dark_orfs.sh`; Pfam status from `scripts/pfam_array.sbatch`. Lineage = MOB
primary cluster (`mob_cluster`), the proxy used throughout this project.

In [ ]:
# The most widespread INDIVIDUAL dark ORF — exact amino-acid sequences, not families.
# Everything upstream of this notebook grouped the dark proteome at 30% identity / 80% coverage
# (scripts/cluster_dark_orfs.sh). Here the unit is the literal protein: two ORFs count as "the
# same" only if their translations are character-for-character identical.
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data" / "plasmidscope_primary").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DOR, PFM, PP = (ROOT / "data" / d for d in ("dark_orf_run", "pfam_run", "plasmidscope_primary"))

S1, S2, S3, S4, NEUT = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#b8b7ae"
INK, INK2, GRID, FAINT = "#0b0b0b", "#52514e", "#dcdbd4", "#d4d3cc"


def tidy(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set(color=GRID, linewidth=1)
    ax.tick_params(colors=INK2, labelsize=9, length=0)
    ax.set_axisbelow(True)


def read_faa(path):
    out, name, cur = {}, None, []
    with open(path) as fh:
        for line in fh:
            if line[0] == ">":
                if name is not None:
                    out[name] = "".join(cur)
                name, cur = line[1:].strip(), []
            else:
                cur.append(line.strip())
    out[name] = "".join(cur)
    return out


PROT = read_faa(DOR / "dark_orfs.faa")                      # ~20 s, 84 MB
BYSEQ = defaultdict(list)
for pid, aa in PROT.items():
    BYSEQ[aa].append(pid)

CL = pd.read_csv(DOR / "dark30_cluster.tsv", sep="\t", header=None, names=["rep", "member"])
MEM2FAM = dict(zip(CL["member"], CL["rep"]))
FAM = (pd.read_csv(DOR / "darkfam_stats.tsv", sep="\t")
       .merge(pd.read_csv(PFM / "darkfam_pfam.tsv", sep="\t")[["rep", "pfam_hit", "pfam_name"]],
              on="rep").set_index("rep"))
MET = pd.read_csv(PP / "plasmid_metadata_master.tsv", sep="\t", low_memory=False,
                  usecols=["plasmid_id", "hab_sub", "mob_cluster", "size_bp"]).set_index("plasmid_id")
ORG = pd.read_csv(PP / "working_set.tsv", sep="\t", low_memory=False,
                  usecols=["plasmid_id", "organism"]).set_index("plasmid_id")["organism"]

NORF, NSEQ = len(PROT), len(BYSEQ)
assert NORF == 378_552, NORF
print(f"dark ORFs {NORF:,}  ->  distinct amino-acid sequences {NSEQ:,} "
      f"({(1 - NSEQ / NORF) * 100:.1f}% are exact duplicates of another ORF)")

## 1. How far does one exact protein sequence travel?

Before ranking anything, the shape of the distribution — because it decides whether "most
widespread" is even a meaningful category.

In [ ]:
# Q1 — how far does a single dark protein sequence actually travel? The unit is the plasmid:
# two copies on one replicon are one plasmid, not two.
spread = pd.Series({s: len({p.rsplit("|", 1)[0] for p in ks}) for s, ks in BYSEQ.items()})
print(f"plasmids per distinct dark sequence: median {spread.median():.0f}  "
      f"mean {spread.mean():.2f}  p99 {spread.quantile(.99):.0f}  max {spread.max():,}")
for k in (1, 2, 10, 20, 50, 100, 500):
    n = int((spread >= k).sum()) if k > 1 else int((spread == 1).sum())
    lab = "on exactly 1 plasmid" if k == 1 else f"on >= {k} plasmids"
    print(f"  {lab:<22}: {n:>7,}  ({n / NSEQ * 100:5.2f}% of distinct sequences)")

# Everything below works on the sequences that travel at all. 20 plasmids is a deliberate floor:
# it is where per-sequence habitat and lineage counts stop being dominated by sampling noise.
MIN_PL = 20
keep = spread[spread >= MIN_PL].index
rows = []
for s in keep:
    ks = BYSEQ[s]
    pl = sorted({p.rsplit("|", 1)[0] for p in ks})
    md = MET.reindex(pl)
    org = ORG.reindex(pl)
    fam = FAM.loc[MEM2FAM[ks[0]]]
    rows.append({"len_aa": len(s), "plasmids": len(pl), "copies": len(ks),
                 "lineages": md["mob_cluster"].nunique(), "habitats": md["hab_sub"].nunique(),
                 "top_hab": md["hab_sub"].value_counts().index[0] if md["hab_sub"].notna().any() else "-",
                 "median_bp": md["size_bp"].median(),
                 "organism": org.value_counts().index[0] if org.notna().any() else "(none named)",
                 "n_org": int(org.notna().sum()),
                 "family_rep": MEM2FAM[ks[0]], "family_size": int(fam["proteins"]),
                 "pfam": fam["pfam_name"] if fam["pfam_hit"] else "", "seq": s})
D = pd.DataFrame(rows).sort_values("plasmids", ascending=False).reset_index(drop=True)
D.insert(0, "orf_id", [f"dORF{i + 1:04d}" for i in range(len(D))])
print(f"\n{len(D):,} distinct sequences occur on >= {MIN_PL} plasmids; they account for "
      f"{D['copies'].sum():,} dark ORFs ({D['copies'].sum() / NORF * 100:.1f}% of the proteome)")
print(f"{(D['pfam'] != '').sum()} of them ({(D['pfam'] != '').mean() * 100:.0f}%) sit in a "
      f"family Pfam can name — so most are unnamed at every level")

In [ ]:
# Figure 1. Grey = sequences seen on one plasmid only; it means "does not travel" and nothing
# else here.
band = pd.cut(spread, [0, 1, 4, 9, 19, 49, 99, 10 ** 9],
              labels=["1", "2-4", "5-9", "10-19", "20-49", "50-99", "100+"]).value_counts().sort_index()
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"wspace": 0.2})

x = np.arange(len(band))
axA.bar(x, band.values, width=0.56, color=[NEUT] + [S1] * (len(band) - 1))
for i, v in enumerate(band.values):
    axA.text(i, v + band.max() * 0.02, f"{v:,}\n{v / NSEQ * 100:.1f}%", ha="center",
             fontsize=8.5, color=INK2)
axA.set(xticks=x, xticklabels=band.index, yticks=[], ylim=(0, band.max() * 1.2))
axA.set_xlabel("distinct plasmids carrying that exact sequence", fontsize=9, color=INK2)
axA.set_title(f"A — an exact dark protein almost never travels\n"
              f"{int(band.loc['1']):,} of {NSEQ:,} sequences ({band.loc['1'] / NSEQ * 100:.0f}%) "
              f"sit on one plasmid", fontsize=10.5, color=INK, loc="left")
tidy(axA)

srt = np.sort(spread.values)[::-1]
axB.plot(np.arange(1, len(srt) + 1), srt, color=S1, linewidth=2)
axB.axhline(MIN_PL, color=GRID, linewidth=1, zorder=0)
axB.text(1.4, MIN_PL * 1.2, f"the {len(D):,} sequences studied below (>= {MIN_PL} plasmids)",
         fontsize=8.5, color=INK2)
axB.scatter([1], [srt[0]], s=30, color=INK, zorder=3)
axB.text(2, srt[0] * 0.86, f"max {srt[0]:,} plasmids", fontsize=9, color=INK)
axB.set(xscale="log", yscale="log", xlim=(1, NSEQ), ylim=(1, srt[0] * 1.6))
axB.set_xlabel("distinct sequences, ranked (log)", fontsize=9, color=INK2)
axB.set_ylabel("plasmids carrying it (log)", fontsize=9, color=INK2)
axB.set_title("B — the same thing as a rank curve\nthe tail is almost the whole distribution",
              fontsize=10.5, color=INK, loc="left")
tidy(axB)

fig.suptitle("How far does one exact dark protein sequence travel?", fontsize=12, color=INK,
             x=0.006, ha="left", y=1.02)
plt.show()

## 2. The naive answer: rank by plasmid count

This is the question as asked. The answer is a specific 83 aa protein — and the columns beside
it are what make the next section necessary.

In [ ]:
# Q2 — the naive answer: rank by how many plasmids carry the exact sequence.
TOP = D.head(15)
print("the 15 most widespread individual dark ORFs, by plasmid count:")
print(TOP[["orf_id", "len_aa", "plasmids", "lineages", "habitats", "median_bp", "top_hab",
           "organism", "pfam"]].to_string(index=False))

w = D.iloc[0]
print(f"\nwinner: {w['orf_id']} — {w['len_aa']} aa on {w['plasmids']:,} plasmids "
      f"({w['copies']:,} copies)")
print(f"  family {w['family_rep']} ({w['family_size']:,} members), "
      f"Pfam: {w['pfam'] if w['pfam'] else 'NO HIT at --cut_ga'}")
print(f"  carriers: median {w['median_bp']:,.0f} bp, dominant habitat {w['top_hab']}, "
      f"host named on {w['n_org']} of {w['plasmids']} (commonest: {w['organism']})")
print(f"\n  BUT it spans only {w['lineages']} MOB primary clusters and "
      f"{w['habitats']} habitats. Hold that thought.")

In [ ]:
# Figure 2. Grey = the sequence's family HAS a Pfam name, so it is explained; slot 1 = unnamed.
t = TOP.iloc[::-1]
y = np.arange(len(t))
cols = [NEUT if p else S1 for p in t["pfam"]]
fig, (axA, axB) = plt.subplots(1, 2, figsize=(14.5, 5.6),
                               gridspec_kw={"wspace": 0.06, "width_ratios": [1.5, 1]})
axA.barh(y, t["plasmids"], height=0.6, color=cols)
for i, (v, p, ln) in enumerate(zip(t["plasmids"], t["pfam"], t["len_aa"])):
    axA.text(v + TOP["plasmids"].max() * 0.012, i, f"{v:,}   {ln} aa   {p or '—'}",
             va="center", fontsize=8.5, color=INK2)
axA.set(yticks=y, yticklabels=t["orf_id"], xticks=[], xlim=(0, TOP["plasmids"].max() * 1.35))
axA.set_xlabel("distinct plasmids carrying the exact sequence", fontsize=9, color=INK2)
axA.set_title(f"A — ranked by plasmid count\ngrey: {(TOP['pfam'] != '').sum()} of 15 sit in a "
              f"Pfam-named family", fontsize=10.5, color=INK, loc="left")
tidy(axA)

axB.barh(y, t["lineages"], height=0.6, color=cols)
for i, v in enumerate(t["lineages"]):
    axB.text(v + 0.25, i, f"{v}", va="center", fontsize=8.5, color=INK2)
axB.set(yticks=y, yticklabels=[], xlim=(0, TOP["lineages"].max() * 1.3), ylim=axA.get_ylim())
axB.set_xlabel("MOB primary clusters spanned", fontsize=9, color=INK2)
axB.set_title("B — the same 15, on lineage spread\nthe scale is 1-12, not 1-800",
              fontsize=10.5, color=INK, loc="left")
tidy(axB)
fig.suptitle("The most widespread individual dark ORFs — and the reason the ranking misleads",
             fontsize=12, color=INK, x=0.006, ha="left", y=0.99)
plt.show()

## 3. Why that ranking is measuring the wrong thing

If the top sequences were independently mobile proteins, they would be found on *different*
plasmids. They are not.

In [ ]:
# Q3 — the trap. If the top sequences were independently mobile proteins they would ride
# different plasmids. Test it directly: how much do their carrier sets overlap?
K = 5
sets = [({p.rsplit("|", 1)[0] for p in BYSEQ[D.loc[i, "seq"]]}) for i in range(K)]
OV = pd.DataFrame(
    [[len(a & b) / min(len(a), len(b)) * 100 for b in sets] for a in sets],
    index=D.loc[:K - 1, "orf_id"], columns=D.loc[:K - 1, "orf_id"])
print(f"carrier-set overlap of the top {K} sequences (% of the smaller set):")
print(OV.round(0).to_string())

core = set.intersection(*sets)
cm = MET.reindex(sorted(core))
print(f"\nplasmids carrying ALL {K}: {len(core):,}")
print(f"  size          median {cm['size_bp'].median():,.0f} bp  "
      f"IQR {cm['size_bp'].quantile(.25):,.0f}-{cm['size_bp'].quantile(.75):,.0f}")
print(f"  MOB clusters  {cm['mob_cluster'].nunique()}")
print(f"  habitats      {cm['hab_sub'].nunique()}   "
      f"dominant {cm['hab_sub'].value_counts().index[0]} "
      f"({cm['hab_sub'].value_counts(normalize=True).iloc[0] * 100:.0f}%)")
print("\nThat is not five widespread proteins. It is ONE small plasmid, sequenced repeatedly:\n"
      "a near-invariant replicon of the same length, in a single MOB lineage. Ranking dark ORFs\n"
      "by plasmid count measures how often a plasmid was sequenced, not how far a protein has\n"
      "travelled — the same confound flagged for lineage spread in dark_families.ipynb.")

In [ ]:
# Figure 3. A sequential ramp for the overlap matrix (a magnitude, not a category), so no
# categorical colour is in play and grey is not used at all in this figure.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 4.6),
                               gridspec_kw={"wspace": 0.28, "width_ratios": [1, 1.15]})
im = axA.imshow(OV.values, cmap="Blues", vmin=0, vmax=100)
for i in range(K):
    for j in range(K):
        v = OV.values[i, j]
        axA.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=9,
                 color="white" if v > 55 else INK)
axA.set(xticks=range(K), yticks=range(K), xticklabels=OV.columns, yticklabels=OV.index)
axA.tick_params(labelsize=8, colors=INK2, length=0)
plt.setp(axA.get_xticklabels(), rotation=35, ha="right")
for s in axA.spines.values():
    s.set_visible(False)
fig.colorbar(im, ax=axA, fraction=0.046, pad=0.04).set_label("% of smaller carrier set",
                                                             fontsize=8.5, color=INK2)
axA.set_title(f"A — the top {K} ride the same plasmids\ncarrier-set overlap",
              fontsize=10.5, color=INK, loc="left")

axB.hist(cm["size_bp"], bins=60, color=S1, edgecolor="white", linewidth=0.3)
axB.set(yticks=[], xlim=(cm["size_bp"].min() * 0.9, cm["size_bp"].max() * 1.1))
axB.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axB.set_xlabel("size of the plasmids carrying all five (bp)", fontsize=9, color=INK2)
axB.set_title(f"B — and they are one replicon\n{len(core):,} plasmids, "
              f"median {cm['size_bp'].median():,.0f} bp, "
              f"{cm['mob_cluster'].nunique()} MOB cluster", fontsize=10.5, color=INK, loc="left")
tidy(axB)
fig.suptitle("Why the naive ranking fails: sequencing depth, not protein mobility",
             fontsize=12, color=INK, x=0.006, ha="left", y=1.02)
plt.show()

## 4. The corrected ranking

Lineage count is the sampling-robust version of "widespread": a protein on 800 clones of one
plasmid spans one lineage, a protein on 40 unrelated plasmids spans forty.

In [ ]:
# Q4 — the corrected ranking. MOB primary cluster is the lineage proxy used throughout this
# project, so "spans N lineages" is the sampling-robust version of "is widespread".
rho = D["plasmids"].corr(D["lineages"], method="spearman")
print(f"Spearman(plasmids, lineages) over the {len(D):,} travelling sequences = {rho:.3f}")
print("Near zero. The two rankings are all but unrelated, so the choice of ranking IS the result.\n")

BYLIN = D.sort_values(["lineages", "plasmids"], ascending=False)
print("most widely DISTRIBUTED individual dark ORFs (by MOB lineages):")
print(BYLIN.head(12)[["orf_id", "len_aa", "plasmids", "lineages", "habitats", "median_bp",
                      "organism", "pfam"]].to_string(index=False))

NEG = BYLIN[BYLIN["pfam"].eq("")]
print(f"\n{len(BYLIN.head(20)[BYLIN.head(20)['pfam'] != ''])} of the top 20 by lineage have a "
      f"Pfam name, and they are the usual backbone: toxin-antitoxin, replication, transposase.\n"
      f"Restricting to the genuinely unnamed:")
print(NEG.head(10)[["orf_id", "len_aa", "plasmids", "lineages", "habitats", "top_hab",
                    "median_bp", "organism"]].to_string(index=False))

for lab, r in (("MOST PLASMIDS", D.iloc[0]), ("MOST LINEAGES", BYLIN.iloc[0]),
               ("MOST LINEAGES, no Pfam name", NEG.iloc[0])):
    print(f"\n{lab}: {r['orf_id']} | {r['len_aa']} aa | {r['plasmids']:,} plasmids | "
          f"{r['lineages']} lineages | {r['habitats']} habitats | "
          f"Pfam {r['pfam'] or 'NONE'}")

In [ ]:
# Figure 4. Grey = Pfam can name the family, same meaning as figure 2. The two axes are the two
# rival definitions of "widespread"; the point of the panel is that they disagree.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13.5, 5),
                               gridspec_kw={"wspace": 0.26, "width_ratios": [1.25, 1]})
for named, col, z in ((True, NEUT, 1), (False, S1, 2)):
    s = D[D["pfam"].ne("") == named]
    axA.scatter(s["plasmids"], s["lineages"], s=18, color=col, alpha=0.75, linewidth=0, zorder=z)
# Label offsets are given as multipliers of the point's own coordinates because both axes are
# log; they are placed so no leader line crosses another label.
for r, txt, fx, fy in ((D.iloc[0], "most plasmids", 0.32, 2.4),
                       (BYLIN.iloc[0], "most lineages", 2.7, 1.35),
                       (NEG.iloc[0], "most lineages,\nstill unnamed", 0.30, 1.7)):
    axA.scatter([r["plasmids"]], [r["lineages"]], s=70, facecolor="none", edgecolor=INK,
                linewidth=1.5, zorder=5)
    axA.annotate(f"{txt}\n{r['orf_id']} ({r['len_aa']} aa)",
                 (r["plasmids"], r["lineages"]), fontsize=8.5, color=INK,
                 xytext=(r["plasmids"] * fx, r["lineages"] * fy),
                 arrowprops=dict(arrowstyle="-", color=INK2, linewidth=0.8))
axA.set(xscale="log", yscale="log", xlim=(MIN_PL * 0.8, D["plasmids"].max() * 2.2),
        ylim=(0.8, D["lineages"].max() * 3))
axA.set_xlabel("plasmids carrying the exact sequence (log)", fontsize=9, color=INK2)
axA.set_ylabel("MOB primary clusters spanned (log)", fontsize=9, color=INK2)
axA.set_title(f"A — the two definitions disagree\nSpearman rho = {rho:.3f} over "
              f"{len(D):,} sequences · grey = Pfam-named", fontsize=10.5, color=INK, loc="left")
tidy(axA)

t = NEG.head(10).iloc[::-1]
y = np.arange(len(t))
axB.barh(y, t["lineages"], height=0.55, color=S1)
for i, (v, p, ln) in enumerate(zip(t["lineages"], t["plasmids"], t["len_aa"])):
    axB.text(v + 0.15, i, f"{v} lineages · {p} plasmids · {ln} aa", va="center", fontsize=8,
             color=INK2)
axB.set(yticks=y, yticklabels=t["orf_id"], xticks=[], xlim=(0, t["lineages"].max() * 1.9))
axB.tick_params(axis="y", labelsize=8)
axB.set_xlabel("MOB primary clusters spanned", fontsize=9, color=INK2)
axB.set_title("B — the genuinely unnamed, best first\nno Pfam hit at any level",
              fontsize=10.5, color=INK, loc="left")
tidy(axB)
fig.suptitle("Widespread by sequencing, or widespread by biology?", fontsize=12, color=INK,
             x=0.006, ha="left", y=1.0)
plt.show()

## 5. The sequences

In [ ]:
# Q5 — the sequences themselves. Written out so they can be BLASTed, folded, or ordered.
OUT = ROOT / "data" / "widespread_orfs"
OUT.mkdir(exist_ok=True)
D.drop(columns="seq").to_csv(OUT / "widespread_dark_orfs.tsv", sep="\t", index=False)
with open(OUT / "widespread_dark_orfs.faa", "w") as fh:
    for _, r in D.iterrows():
        fh.write(f">{r['orf_id']} plasmids={r['plasmids']} lineages={r['lineages']} "
                 f"habitats={r['habitats']} len={r['len_aa']} "
                 f"pfam={r['pfam'] or 'none'}\n{r['seq']}\n")
print(f"wrote {len(D):,} sequences -> {(OUT / 'widespread_dark_orfs.faa').relative_to(ROOT)}")
print(f"      metadata            -> {(OUT / 'widespread_dark_orfs.tsv').relative_to(ROOT)}\n")


def show(r, label):
    print("=" * 78)
    print(f"{label}\n{r['orf_id']}   {r['len_aa']} aa")
    print(f"  on {r['plasmids']:,} plasmids ({r['copies']:,} copies), {r['lineages']} MOB "
          f"lineages, {r['habitats']} habitats")
    print(f"  carriers: median {r['median_bp']:,.0f} bp, dominant habitat {r['top_hab']}")
    print(f"  host named on {r['n_org']} of {r['plasmids']}; commonest {r['organism']}")
    print(f"  30%-family {r['family_rep']} ({r['family_size']:,} members)")
    print(f"  Pfam-A 38.2 --cut_ga: {r['pfam'] or 'NO HIT'}")
    print(f">{r['orf_id']}")
    for i in range(0, len(r["seq"]), 60):
        print(r["seq"][i:i + 60])
    print()


show(D.iloc[0], "MOST PLASMIDS — the headline answer (but see section 3)")
show(BYLIN.iloc[0], "MOST MOB LINEAGES — the sampling-robust answer")
show(NEG.iloc[0], "MOST MOB LINEAGES WITH NO PFAM NAME — the one worth working on")

## Answers

**Ranked by plasmid count** — `dORF0001`, **83 aa, on 820 plasmids** (823 copies). No PlasAnn
call, and its 994-member family has no Pfam-A hit at `--cut_ga`. Carriers are ~4.25 kb,
77% `Human: unspecified`, and where a host is named at all it is *Bacteroides thetaiotaomicron*.

```
MEALSVREYRNNLAASFTKADNGEQVLIRRKNEIYALVKVGREDLMITPELQARIDKAREEIKSGKCVTLKSSEDIDAYFDSL
```

**But that number is an artifact.** The top five sequences overlap on 82-98% of their carriers,
and 290 plasmids carry all five — median 4,247 bp, IQR 4,225-4,247, **one MOB cluster**, 100%
with no replicon call. That is a single ~4.25 kb *Bacteroides* cryptic plasmid sequenced roughly
800 times. Plasmid count measures sequencing effort, not protein mobility.

**Ranked by lineage spread** — `dORF0105`, **92 aa on 79 plasmids across 24 MOB lineages** and
11 habitats. Spearman between the two rankings is **0.077**, so this is a different object, not
a re-ordering. Its family *does* have a Pfam hit — `PhdYeFM_antitox` — so it is a known
antitoxin PlasAnn failed to call, not new biology.

**The one actually worth working on** — `dORF0350`, **75 aa, 41 plasmids, 11 MOB lineages, 12
habitats**, commonest host *Acinetobacter baumannii*, median carrier 8.6 kb. No PlasAnn call, no
Pfam name for its 57-member family, and spread across enough independent lineages that sampling
cannot explain it.

```
LLKVNDLIAKSKNGTEIIVSLIPLHKMQNTRQGLKQIEVGKRVLLESGIEVDLNLDGRTFYTSLNQLFKLNHKVA
```

### The general lesson

Of the 20 most lineage-spread dark ORFs, nine sit in Pfam-named families, and the names are
`PhdYeFM_antitox`, `ParE_toxin`, `YafQ_toxin`, `MazE_antitoxin`, `RelB`, `HTH_Tnp_1` — toxin-
antitoxin systems and transposases. The same pattern as `dark_families.ipynb` found at family
level: **the most widespread dark proteins are mostly annotation failures, and the genuinely
novel ones are smaller and less common.** Any ranking that does not correct for lineage will
return the most-sequenced plasmid rather than the most-travelled protein.

### Limits

75.6% of distinct dark sequences occur on exactly one plasmid, so this analysis speaks only for
the 1,107 that reach 20 plasmids — 0.5% of distinct sequences. Exact-identity grouping is also
brittle by construction: a single substitution splits a sequence in two, so these counts are
lower bounds on real distribution. That is what the 30%-family view exists to handle, and the
two should be read together.